# TEST INFERENCE PIPELINE

In [ ]:
from pathlib import Path
import os
import numpy as np
import torch
import io
from PIL import Image

from torch.utils.data import DataLoader
from emma import EmmaDataset, EmmaSearcher, SearchQuery
from emma.dataloader import EmmaMapDataset
from emma.dataloader import emma_collate

from transformers import DepthProImageProcessor, DepthProForDepthEstimation

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    print("Ready")
    device = torch.device('cuda')
else:
    print("Not Ready")
    device = torch.device('cpu')

## > DATASET

In [ ]:
dataset_root = None  # e.g. Path("/path/to/EMMa")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the EMMa dataset.")

dataset_root = Path(dataset_root)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")

In [ ]:
# INIT DATASET
ds = EmmaDataset(dataset_root)

In [ ]:
# QUERY
se = EmmaSearcher(ds)
query = SearchQuery(splits=["Test"], require_listed=["mass", "category"], require_present=["package_dimensions"], quantity_range=[1,1], limit=10)
asins = se.query_asins(query)
print(len(asins))

In [ ]:
def my_transform(asin: str, rec: dict):
    image_dict = ds.get_images(rec)

    first_img_key, first_img_bytes = next(iter(image_dict.items()))
    first_img = Image.open(io.BytesIO(first_img_bytes)).convert("RGB")
    first_img = np.array(first_img)

    dimensions = np.squeeze(ds.get_package_dimensions(rec)) * 2.54 / 100.0
    normalization = np.float32(np.linalg.norm(dimensions))

    return {
        "image": first_img,
        "normalization": normalization,
        "asin": asin,
    }

In [ ]:
batch_size = 2

loader = DataLoader(
    EmmaMapDataset(ds, asins, transform=my_transform),
    batch_size=batch_size, shuffle=False, collate_fn=emma_collate,
)

## > MODEL

In [ ]:
depth_pro_root = None  # e.g. Path("/path/to/Depth-Pro")

if depth_pro_root is None:
    raise ValueError("Please set `depth_pro_root` to the path of the Depth Pro model.")

depth_pro_root = Path(depth_pro_root)

if not depth_pro_root.exists():
    raise FileNotFoundError(f"Model path does not exist: {depth_pro_root}")

In [ ]:
image_processor = DepthProImageProcessor.from_pretrained(depth_pro_root)
model = DepthProForDepthEstimation.from_pretrained(depth_pro_root, torch_dtype=torch.float16).to(device)

## > INFERENCE

In [ ]:
inference_results = []

for batch_idx, batch in enumerate(loader):
    images = batch["image"]

    inputs = image_processor(images=images, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = [(img.shape[0], img.shape[1]) for img in images]

    post_processed_output = image_processor.post_process_depth_estimation(
        outputs, target_sizes=target_sizes,
    )

    for prediction in post_processed_output:
        inference_results.append({
            "depth": prediction["predicted_depth"].detach().cpu().numpy(),
            "focal_length": prediction["focal_length"].detach().cpu().numpy(),
            "field_of_view": prediction["field_of_view"].detach().cpu().numpy()
        })

## > VISUALIZE

In [ ]:
for result in inference_results[:10]:
    
    depth = result["depth"]
    depth_img = (depth - depth.min()) / (depth.max() - depth.min())
    depth_img = depth_img * 255.
    depth_img = Image.fromarray(depth_img.astype("uint8"))

    print("field_of_view: ", result["field_of_view"])
    print("focal_length: ", result["focal_length"])
    depth_img.show()